# 3_12 Train multi-class classifier (linear Softmax)

Lean runnable training notebook (no slide deck).

Three touching 2D blobs (**cat / dog / pig**). Model is a linear map
`R^2 -> 3` logits + cross-entropy (Softmax inside the loss).

| Piece | Role |
|-------|------|
| data | Gaussian blobs that touch / slightly overlap |
| model | `nn.Linear(2, 3)` |
| loss | `F.cross_entropy` |
| loop | SGD + mini-batches; monitor val; report test once |


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(45)
np.random.seed(45)

CLASS_NAMES = ["cat", "dog", "pig"]
N_PER, BLOB_STD = 120, 0.62
CENTERS = {
    "cat": (-1.05, -0.55),
    "dog": ( 1.05, -0.55),
    "pig": ( 0.00,  1.05),
}
EPOCHS, LR, BATCH_SIZE = 40, 0.20, 32

X_list, y_list = [], []
for label, (_name, mean) in enumerate(CENTERS.items()):
    pts = torch.tensor(mean) + BLOB_STD * torch.randn(N_PER, 2)
    X_list.append(pts)
    y_list.append(torch.full((N_PER,), label, dtype=torch.long))
X = torch.cat(X_list, dim=0)
y = torch.cat(y_list, dim=0)
perm = torch.randperm(len(y))
X, y = X[perm], y[perm]
print(X.shape, y.shape, CLASS_NAMES)


## Model

One linear score per class. Softmax turns scores into a distribution;
prediction = argmax.


In [ ]:
class SoftmaxClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 3)  # 2 features -> 3 logits

    def forward(self, x):
        return self.fc(x)          # raw scores


def dramatic_init_(model: SoftmaxClassifier) -> None:
    """Deliberately wrong, high-confidence init so boundaries move a lot."""
    with torch.no_grad():
        W = torch.tensor(
            [
                [0.0, 5.0],    # "cat" fires on +x2  (pig's home)
                [-5.0, -1.5],  # "dog" fires on -x1  (cat's home)
                [5.0, -1.5],   # "pig" fires on +x1  (dog's home)
            ],
            dtype=torch.float32,
        )
        joint = torch.tensor([0.65, 0.40], dtype=torch.float32)
        b = -W @ joint
        model.fc.weight.copy_(W)
        model.fc.bias.copy_(b)


n = len(y)
n_train, n_val = int(0.60 * n), int(0.20 * n)
X_tr, y_tr = X[:n_train], y[:n_train]
X_va, y_va = X[n_train:n_train + n_val], y[n_train:n_train + n_val]
X_te, y_te = X[n_train + n_val:], y[n_train + n_val:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False)

torch.manual_seed(7)
model = SoftmaxClassifier()
dramatic_init_(model)
opt = torch.optim.SGD(model.parameters(), lr=LR)
print(model)


## Training loop

```text
logits -> cross_entropy -> zero_grad -> backward -> step
```


In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total, n_ex, correct = 0.0, 0, 0
    for xb, yb in loader:
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        total += loss.item() * len(yb)
        n_ex += len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
    return total / n_ex, correct / n_ex


for epoch in range(EPOCHS):
    tr, tr_acc = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va, va_acc = run_epoch(val_loader, train=False)
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(
            f"epoch {epoch+1:02d}/{EPOCHS}  "
            f"train={tr:.3f}({tr_acc:.3f})  val={va:.3f}({va_acc:.3f})"
        )


## Inference + one-shot test report


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_te[:5])
    prob = torch.softmax(logits, dim=1)
    pred = prob.argmax(dim=1)
    for i in range(5):
        print(
            f"true={CLASS_NAMES[int(y_te[i])]:3s}  "
            f"pred={CLASS_NAMES[int(pred[i])]:3s}  "
            f"p={prob[i].numpy().round(3)}"
        )
    te_loss, te_acc = run_epoch(test_loader, train=False)
print(f"test CE={te_loss:.3f}  acc={te_acc:.3f}")
# Do not tune LR / epochs on this number.
